# Lucknow Forecasting Model Comparison

This notebook compares the existing LSTM results with classical baselines and additional deep-learning models.

All models use the same targets: `CLOUD_Vertical_Visibility` and `VIS_Visibility_MOR`. The comparison uses a chronological 70%/15%/15% train/validation/test split and MAE, RMSE, and R2. No range-aware weighting or range-aware target prediction is used.

In [1]:
from pathlib import Path
import copy
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE_DIR = Path(r"D:\IMD\Lucknow")
DATA_FILE = BASE_DIR / "lucknow_worked" / "07_LSTM_READY_DATA" / "lucknow_unified_lstm_dataset_aug2024_dec2025.csv"
RESULTS_DIR = BASE_DIR / "result" / "benchmark_lucknow_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMNS = ["CLOUD_Vertical_Visibility", "VIS_Visibility_MOR"]
TIMESTAMP_COLUMN = "Datetime"
LOOKBACK = 96
HORIZON = 1
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
SEED = 42
MAX_TRAIN_ROWS = 150_000
MAX_DEEP_EVALUATION_ROWS = 150_000
DEEP_EPOCHS = 12
BATCH_SIZE = 512
PATIENCE = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

Device: cuda
PyTorch: 2.5.1


In [2]:
df = pd.read_csv(DATA_FILE, low_memory=False)
df[TIMESTAMP_COLUMN] = pd.to_datetime(df[TIMESTAMP_COLUMN], errors="coerce")
df = df.dropna(subset=[TIMESTAMP_COLUMN] + TARGET_COLUMNS).drop_duplicates(subset=[TIMESTAMP_COLUMN]).sort_values(TIMESTAMP_COLUMN).reset_index(drop=True)

feature_columns = df.select_dtypes(include=[np.number]).columns.tolist()
n_rows = len(df)
train_end = int(n_rows * TRAIN_RATIO)
validation_end = int(n_rows * (TRAIN_RATIO + VALIDATION_RATIO))

train_medians = df.iloc[:train_end][feature_columns].median().fillna(0.0)
df[feature_columns] = df[feature_columns].replace([np.inf, -np.inf], np.nan).fillna(train_medians)

train_frame = df.iloc[:train_end].copy()
validation_frame = df.iloc[train_end:validation_end].copy()
test_frame = df.iloc[validation_end:].copy()

feature_scaler = StandardScaler()
target_scaler = StandardScaler()
feature_scaler.fit(train_frame[feature_columns])
target_scaler.fit(train_frame[TARGET_COLUMNS])

def transform_frame(frame):
    features = feature_scaler.transform(frame[feature_columns]).astype(np.float32)
    targets = target_scaler.transform(frame[TARGET_COLUMNS]).astype(np.float32)
    return features, targets

train_features, train_targets = transform_frame(train_frame)
validation_features, validation_targets = transform_frame(validation_frame)
test_features, test_targets = transform_frame(test_frame)

def make_sequences(features, targets, lookback=LOOKBACK, horizon=HORIZON):
    last_start = len(features) - lookback - horizon + 1
    if last_start <= 0:
        return np.empty((0, lookback, features.shape[1]), np.float32), np.empty((0, targets.shape[1]), np.float32)
    x = np.empty((last_start, lookback, features.shape[1]), dtype=np.float32)
    y = np.empty((last_start, targets.shape[1]), dtype=np.float32)
    for i in range(last_start):
        x[i] = features[i:i + lookback]
        y[i] = targets[i + lookback + horizon - 1]
    return x, y

train_features_deep = train_features[-(MAX_TRAIN_ROWS + LOOKBACK + HORIZON - 1):]
train_targets_deep = train_targets[-(MAX_TRAIN_ROWS + LOOKBACK + HORIZON - 1):]
validation_features_deep = validation_features[-(MAX_DEEP_EVALUATION_ROWS + LOOKBACK + HORIZON - 1):]
validation_targets_deep = validation_targets[-(MAX_DEEP_EVALUATION_ROWS + LOOKBACK + HORIZON - 1):]
test_features_deep = test_features[-(MAX_DEEP_EVALUATION_ROWS + LOOKBACK + HORIZON - 1):]
test_targets_deep = test_targets[-(MAX_DEEP_EVALUATION_ROWS + LOOKBACK + HORIZON - 1):]

X_train, y_train = make_sequences(train_features_deep, train_targets_deep)
X_validation, y_validation = make_sequences(validation_features_deep, validation_targets_deep)
X_test, y_test = make_sequences(test_features_deep, test_targets_deep)

print("Rows:", n_rows)
print("Features:", len(feature_columns), feature_columns)
print("Deep-learning sequence shapes:", X_train.shape, X_validation.shape, X_test.shape)

Rows: 2520761
Features: 22 ['CLOUD_Cloud_Base_1', 'CLOUD_Cloud_Base_2', 'CLOUD_Cloud_Base_3', 'CLOUD_SITE', 'CLOUD_OCTA1', 'CLOUD_OCTA2', 'CLOUD_OCTA3', 'CLOUD_OCTA4', 'CLOUD_Vertical_Visibility', 'PTU_Pressure_QNH', 'PTU_Temperature', 'PTU_DewPoint', 'PTU_Humidity', 'PW_Rain_Intensity', 'PW_Rain_Sum', 'VIS_Visibility_MOR', 'VIS_RVR', 'WIND_Wind_Dir', 'WIND_Wind_Speed', 'WIND_Wind_Gust', 'Hour_Sin', 'Hour_Cos']
Deep-learning sequence shapes: (150000, 96, 22) (150000, 96, 22) (150000, 96, 22)


## Classical baselines and machine-learning models

In [3]:
def compact_history(features, targets):
    values = features
    prefix = np.vstack([np.zeros((1, values.shape[1]), dtype=np.float64), np.cumsum(values.astype(np.float64), axis=0)])
    prefix_sq = np.vstack([np.zeros((1, values.shape[1]), dtype=np.float64), np.cumsum(values.astype(np.float64) ** 2, axis=0)])
    starts = np.arange(len(values) - LOOKBACK - HORIZON + 1)
    ends = starts + LOOKBACK
    sums = prefix[ends] - prefix[starts]
    sqs = prefix_sq[ends] - prefix_sq[starts]
    means = sums / LOOKBACK
    stds = np.sqrt(np.maximum(sqs / LOOKBACK - means ** 2, 0.0))
    latest = values[ends - 1]
    x = np.concatenate([latest, means, stds], axis=1).astype(np.float32)
    y = targets[LOOKBACK + HORIZON - 1:LOOKBACK + HORIZON - 1 + len(x)]
    target_indices = [feature_columns.index(t) for t in TARGET_COLUMNS]
    latest_original = feature_scaler.inverse_transform(latest)
    persistence = latest_original[:, target_indices]
    return x, y, persistence

def metric_rows(model_name, split_name, actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    rows = []
    for i, target in enumerate(TARGET_COLUMNS):
        rows.append({
            "model": model_name,
            "category": "Machine learning" if model_name not in ("persistence", "training_mean") else "Baseline",
            "split": split_name,
            "target": target,
            "MAE": mean_absolute_error(actual[:, i], predicted[:, i]),
            "RMSE": mean_squared_error(actual[:, i], predicted[:, i]) ** 0.5,
            "R2": r2_score(actual[:, i], predicted[:, i]),
        })
    return rows

train_x, train_y, _ = compact_history(train_features, train_targets)
validation_x, validation_y, validation_persistence = compact_history(validation_features, validation_targets)
test_x, test_y, test_persistence = compact_history(test_features, test_targets)

if MAX_TRAIN_ROWS > 0 and len(train_x) > MAX_TRAIN_ROWS:
    train_x = train_x[-MAX_TRAIN_ROWS:]
    train_y = train_y[-MAX_TRAIN_ROWS:]

train_y_actual = target_scaler.inverse_transform(train_y)
validation_y_actual = target_scaler.inverse_transform(validation_y)
test_y_actual = target_scaler.inverse_transform(test_y)
validation_persistence_actual = validation_persistence
test_persistence_actual = test_persistence

results = []
results += metric_rows("persistence", "validation", validation_y_actual, validation_persistence_actual)
results += metric_rows("persistence", "test", test_y_actual, test_persistence_actual)

train_mean = train_y_actual.mean(axis=0)
results += metric_rows("training_mean", "validation", validation_y_actual, np.broadcast_to(train_mean, validation_y_actual.shape))
results += metric_rows("training_mean", "test", test_y_actual, np.broadcast_to(train_mean, test_y_actual.shape))

models = {
    "linear_regression": make_pipeline(StandardScaler(), LinearRegression()),
    "ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "extra_trees": ExtraTreesRegressor(n_estimators=100, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
    "random_forest": RandomForestRegressor(n_estimators=100, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
    "hist_gradient_boosting": MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=150, learning_rate=0.08, l2_regularization=1.0, random_state=SEED)),
}

for name, model in models.items():
    print("Fitting", name)
    model.fit(train_x, train_y_actual)
    results += metric_rows(name, "validation", validation_y_actual, model.predict(validation_x))
    results += metric_rows(name, "test", test_y_actual, model.predict(test_x))

Fitting linear_regression
Fitting ridge
Fitting extra_trees
Fitting random_forest
Fitting hist_gradient_boosting


## Deep-learning models

The sequence models use 96-step windows and are trained with early stopping on the validation period. The benchmark includes MLP, 1D CNN, GRU, LSTM, BiLSTM, TCN, Attention-LSTM, Transformer encoder, PatchTST-style, TimesNet-style, and TFT-style architectures.

In [4]:
class MLPForecaster(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(), nn.Linear(input_size, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, output_size),
        )

    def forward(self, x):
        return self.network(x)


class CNNForecaster(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(input_size, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, output_size),
        )

    def forward(self, x):
        return self.network(x.transpose(1, 2))


class GRUForecaster(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.gru = nn.GRU(input_size, 64, num_layers=2, batch_first=True, dropout=0.2)
        self.output = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, output_size))

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.output(hidden[-1])


class LSTMForecaster(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, 128, batch_first=True)
        self.dropout1 = nn.Dropout(0.2)
        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.dropout2 = nn.Dropout(0.2)
        self.network = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, output_size))

    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.dropout1(x)
        x, _ = self.lstm2(x)
        x = self.dropout2(x[:, -1, :])
        return self.network(x)


class BiLSTMForecaster(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, 64, batch_first=True, bidirectional=True)
        self.network = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, output_size),
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        return self.network(x[:, -1, :])


class TemporalBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        residual = x
        x = torch.relu(self.conv1(x))
        x = self.dropout(x)
        x = self.conv2(x)
        return torch.relu(x + residual)


class TCNForecaster(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.input_conv = nn.Conv1d(input_size, 64, kernel_size=1)
        self.blocks = nn.ModuleList([TemporalBlock(64, dilation) for dilation in (1, 2, 4, 8, 16)])
        self.output = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, output_size),
        )

    def forward(self, x):
        x = self.input_conv(x.transpose(1, 2))
        for block in self.blocks:
            x = block(x)
        return self.output(x)


class AttentionLSTMForecaster(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, 96, batch_first=True)
        self.attention = nn.Sequential(nn.Linear(96, 48), nn.Tanh(), nn.Linear(48, 1))
        self.output = nn.Sequential(
            nn.Linear(96, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, output_size),
        )

    def forward(self, x):
        sequence, _ = self.lstm(x)
        weights = torch.softmax(self.attention(sequence), dim=1)
        context = torch.sum(sequence * weights, dim=1)
        return self.output(context)


class TransformerForecaster(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.input_projection = nn.Linear(input_size, 128)
        layer = nn.TransformerEncoderLayer(
            d_model=128, nhead=8, dim_feedforward=256, dropout=0.1, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=3)
        self.output = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, output_size))

    def forward(self, x):
        x = self.encoder(self.input_projection(x))
        return self.output(x[:, -1, :])


class PatchTSTForecaster(nn.Module):
    def __init__(self, input_size, output_size, lookback=96, patch_size=16):
        super().__init__()
        self.patch_size = patch_size
        self.patch_count = lookback // patch_size
        self.patch_projection = nn.Linear(patch_size * input_size, 128)
        layer = nn.TransformerEncoderLayer(
            d_model=128, nhead=8, dim_feedforward=256, dropout=0.1, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.output = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, output_size))

    def forward(self, x):
        patches = x.unfold(1, self.patch_size, self.patch_size)
        patches = patches.permute(0, 1, 3, 2).reshape(
            x.shape[0], self.patch_count, -1,
        )
        encoded = self.encoder(self.patch_projection(patches))
        return self.output(encoded.mean(dim=1))


class TimesNetForecaster(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.period_branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(input_size, 64, kernel_size=5, padding=2),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(1),
            )
            for _ in range(3)
        ])
        self.output = nn.Sequential(
            nn.Linear(192, 64), nn.ReLU(), nn.Linear(64, output_size),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        branches = [branch(x).flatten(1) for branch in self.period_branches]
        return self.output(torch.cat(branches, dim=1))


class GatedResidualNetwork(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(size, size), nn.ELU(), nn.Linear(size, size), nn.Sigmoid(),
        )

    def forward(self, x):
        gate = self.network(x)
        return gate * x + (1.0 - gate) * torch.tanh(x)


class TFTStyleForecaster(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.variable_selection = nn.Sequential(
            nn.Linear(input_size, input_size), nn.Softmax(dim=-1),
        )
        self.grn = GatedResidualNetwork(input_size)
        valid_heads = [n for n in range(1, min(input_size, 8) + 1) if input_size % n == 0]
        nhead = max(valid_heads)
        layer = nn.TransformerEncoderLayer(
            d_model=input_size, nhead=nhead, dim_feedforward=128,
            dropout=0.1, batch_first=True,
        )
        self.temporal_encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.output = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Linear(64, output_size),
        )

    def forward(self, x):
        weights = self.variable_selection(x)
        x = self.grn(x * weights)
        x = self.temporal_encoder(x)
        return self.output(x[:, -1, :])


In [5]:
def inverse_target(predicted_scaled):
    return target_scaler.inverse_transform(np.asarray(predicted_scaled))

def predict_batches(model, features):
    model.eval()
    predictions = []
    with torch.no_grad():
        loader = DataLoader(torch.from_numpy(features), batch_size=BATCH_SIZE, shuffle=False)
        for batch in loader:
            predictions.append(model(batch.to(DEVICE)).cpu().numpy())
    return np.concatenate(predictions, axis=0)

def train_deep_model(name, model):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)), batch_size=BATCH_SIZE, shuffle=True)
    validation_loader = DataLoader(TensorDataset(torch.from_numpy(X_validation), torch.from_numpy(y_validation)), batch_size=BATCH_SIZE, shuffle=False)
    best_loss = float("inf")
    best_state = None
    wait = 0

    for epoch in range(DEEP_EPOCHS):
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        validation_loss = 0.0
        items = 0
        with torch.no_grad():
            for inputs, targets in validation_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                loss = criterion(model(inputs), targets)
                validation_loss += loss.item() * len(inputs)
                items += len(inputs)
        validation_loss /= max(items, 1)
        print(f"{name:12s} epoch {epoch + 1:02d}/{DEEP_EPOCHS} val_loss={validation_loss:.6f}")
        if validation_loss < best_loss:
            best_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(name, "early stopping")
                break

    model.load_state_dict(best_state)
    validation_prediction = inverse_target(predict_batches(model, X_validation))
    test_prediction = inverse_target(predict_batches(model, X_test))
    validation_actual = inverse_target(y_validation)
    test_actual = inverse_target(y_test)
    return validation_actual, test_actual, validation_prediction, test_prediction

deep_models = {
    "MLP": MLPForecaster(X_train.shape[1] * X_train.shape[2], len(TARGET_COLUMNS)),
    "1D_CNN": CNNForecaster(X_train.shape[2], len(TARGET_COLUMNS)),
    "GRU": GRUForecaster(X_train.shape[2], len(TARGET_COLUMNS)),
    "LSTM_trained": LSTMForecaster(X_train.shape[2], len(TARGET_COLUMNS)),
    "BiLSTM": BiLSTMForecaster(X_train.shape[2], len(TARGET_COLUMNS)),
    "TCN": TCNForecaster(X_train.shape[2], len(TARGET_COLUMNS)),
    "Attention_LSTM": AttentionLSTMForecaster(X_train.shape[2], len(TARGET_COLUMNS)),
    "Transformer": TransformerForecaster(X_train.shape[2], len(TARGET_COLUMNS)),
    "PatchTST_style": PatchTSTForecaster(X_train.shape[2], len(TARGET_COLUMNS)),
    "TimesNet_style": TimesNetForecaster(X_train.shape[2], len(TARGET_COLUMNS)),
    "TFT_style": TFTStyleForecaster(X_train.shape[2], len(TARGET_COLUMNS)),
}

for name, model in deep_models.items():
    validation_actual, test_actual, validation_prediction, test_prediction = train_deep_model(name, model)
    results += metric_rows(name, "validation", validation_actual, validation_prediction)
    results += metric_rows(name, "test", test_actual, test_prediction)

MLP          epoch 01/12 val_loss=0.034024
MLP          epoch 02/12 val_loss=0.032770
MLP          epoch 03/12 val_loss=0.035565
MLP          epoch 04/12 val_loss=0.034996
MLP          epoch 05/12 val_loss=0.035210
MLP          epoch 06/12 val_loss=0.034326
MLP          epoch 07/12 val_loss=0.032865
MLP          epoch 08/12 val_loss=0.031413
MLP          epoch 09/12 val_loss=0.032436
MLP          epoch 10/12 val_loss=0.030142
MLP          epoch 11/12 val_loss=0.027616
MLP          epoch 12/12 val_loss=0.031795
1D_CNN       epoch 01/12 val_loss=0.052444
1D_CNN       epoch 02/12 val_loss=0.056049
1D_CNN       epoch 03/12 val_loss=0.060151
1D_CNN       epoch 04/12 val_loss=0.068598
1D_CNN       epoch 05/12 val_loss=0.074411
1D_CNN       epoch 06/12 val_loss=0.082659
1D_CNN       epoch 07/12 val_loss=0.090084
1D_CNN       epoch 08/12 val_loss=0.090513
1D_CNN       epoch 09/12 val_loss=0.097490
1D_CNN early stopping
GRU          epoch 01/12 val_loss=0.016724
GRU          epoch 02/12 val_los

## Existing LSTM result

This adds the previously executed LSTM test metrics from `result/train_lucknow_lstm/evaluation_metrics.csv`, so the comparison includes the actual existing run rather than a duplicated or newly guessed value.

In [6]:
existing_lstm = pd.read_csv(BASE_DIR / "result" / "train_lucknow_lstm" / "evaluation_metrics.csv")
existing_lstm = existing_lstm[existing_lstm["split"] == "test"].copy()
for _, row in existing_lstm.iterrows():
    results.append({
        "model": "LSTM_existing_run",
        "category": "Deep learning",
        "split": "test",
        "target": row["target"],
        "MAE": row["MAE"],
        "RMSE": row["RMSE"],
        "R2": row["R2"],
    })

## Combined test ranking

In [7]:
metrics = pd.DataFrame(results)
test_metrics = metrics[metrics["split"] == "test"].sort_values(["target", "RMSE"]).reset_index(drop=True)
test_metrics.to_csv(RESULTS_DIR / "test_metrics_ranked.csv", index=False)
metrics.to_csv(RESULTS_DIR / "model_comparison_metrics.csv", index=False)

display(test_metrics)
print("\nBest test model by target:")
best = test_metrics.loc[test_metrics.groupby("target")["RMSE"].idxmin()]
display(best[["target", "model", "category", "MAE", "RMSE", "R2"]].sort_values("target"))

,model,category,split,target,MAE,RMSE,R2
0,persistence,Baseline,test,CLOUD_Vertical_Visibility,12.940503,404.065376,0.984789
1,linear_regression,Machine learning,test,CLOUD_Vertical_Visibility,59.192928,430.884265,0.982702
2,ridge,Machine learning,test,CLOUD_Vertical_Visibility,59.326096,430.954138,0.982697
3,LSTM_existing_run,Deep learning,test,CLOUD_Vertical_Visibility,127.280609,602.309565,0.966201
4,BiLSTM,Machine learning,test,CLOUD_Vertical_Visibility,163.365158,624.501676,0.976971
5,Transformer,Machine learning,test,CLOUD_Vertical_Visibility,217.341980,794.153913,0.962760
6,TCN,Machine learning,test,CLOUD_Vertical_Visibility,276.599945,912.452910,0.950839
7,GRU,Machine learning,test,CLOUD_Vertical_Visibility,261.193726,921.229342,0.949889
8,LSTM_trained,Machine learning,test,CLOUD_Vertical_Visibility,305.132385,1024.552951,0.938018
9,TFT_style,Machine learning,test,CLOUD_Vertical_Visibility,327.119202,1124.709185,0.925307



Best test model by target:


,target,model,category,MAE,RMSE,R2
0,CLOUD_Vertical_Visibility,persistence,Baseline,12.940503,404.065376,0.984789
19,VIS_Visibility_MOR,ridge,Machine learning,53.307125,173.285840,0.997580
